In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]  # repository root; run from research/notebooks with pidu installed

In [ ]:
import numpy as np
import os
from pathlib import Path

dir_path = (ROOT / 'data/experimental_data/metastructure')

data = {}

In [ ]:
# None, 9, 12, 14
sample = 14

meas = np.loadtxt(dir_path / f"{str(sample) if sample is not None else 'si'}" / f"{f'm{sample}' if sample is not None else 'si'}_power.dat", skiprows=1)
fit = np.loadtxt(dir_path / f"{str(sample) if sample is not None else 'si'}" / f"{f'm{sample}' if sample is not None else 'si'}_fit.dat", skiprows=1)

print(f"Meas: {meas.shape}, Fit: {fit.shape}")

In [ ]:
# plot signals
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2,1)

ax[0].plot(meas[:,0], meas[:,1], label='Measured')
ax[1].plot(fit[:,0], fit[:,1], label='Fitted')

ax[0].set(
  ylim=(0, 1.1),
)

ax[1].set(
  ylim=(0, 1.1),
)

In [ ]:
# Interpolate
import numpy as np
from scipy.interpolate import interp1d

f_interp = interp1d(fit[:,0], fit[:,1], kind='linear', fill_value="extrapolate")
si_fit_interp = f_interp(meas[:,0])

residual = meas[:,1] - si_fit_interp

In [ ]:
# plot diff
import matplotlib.pyplot as plt

fig, ax = plt.subplots(3,1)

ax[0].plot(fit[:,0], fit[:,1], label='Fitted')          # 
ax[0].plot(meas[:,0], meas[:,1], label='Measured')      # 
ax[0].plot(meas[:,0], np.abs(residual), label='Residual')  # 
ax[1].plot(meas[:,0], residual, label='Residual')          #
ax[2].plot(meas[:,0], residual/si_fit_interp, label='Abs Residual')  #
ax[1].axhline(np.mean(residual[15:220]), color='r', linestyle='--', label='Mean Residual')

print(np.mean(residual[15:220]), np.std(residual[15:220]), meas[220,0])

ax[0].set(
  ylim=(0, 1.1),
  xlim=(0, 120),
)
ax[1].set(
  ylim=(-0.4, 0.4),
  xlim=(0, 120),
)
ax[2].set(
  ylim=(-0.4, 0.4),
  xlim=(0, 120),
)

data_to_export = np.column_stack((meas[:,0], meas[:,1], si_fit_interp, np.abs(residual), residual/si_fit_interp))
export_path = dir_path / f"{str(sample) if sample is not None else 'si'}_residuals.dat"
np.savetxt(export_path, data_to_export, header="Wavenumbers (cm-1) Si(meas) Si(fit) Residual Per_Residual", fmt="%.6e", comments="")

print(f"Saved data to {export_path}")

data[str(sample) if sample is not None else 'si'] = data_to_export

In [ ]:
# DOES NOT WORK
# linefitting of residuals
from scipy.optimize import curve_fit

def f_noise(x, a, c, d, e):
    return a*np.cos(2*np.pi*c*x + d) + e
  
# fit residual
popt, pcov = curve_fit(
  f_noise, 
  meas[15:220,0], 
  residual[15:220], 
  p0=[5, 1.5, 90, 0],
  )
print("Fitted parameters:", popt)

# plot fitted noise
fig, ax = plt.subplots(1,1)
ax.plot(meas[:,0], residual, label='Residual')
ax.plot(meas[:,0], f_noise(meas[:,0], *popt), label='Fitted Noise', linestyle='--')
# ax.plot(si_meas[:,0], f_noise(si_meas[:,0], 0.1, 1, 1, 0), label='Fitted Noise', linestyle='--')
ax.set(
  ylim=(-0.4, 0.4),
  xlim=(0, 120),
)

In [ ]:
# Create difference matrix (.xlsx) with [N_samples, N_wavenumbers]
import pandas as pd
fig, ax = plt.subplots(1,1)
xlsx_to_export = np.zeros((len(data)+1, list(data.values())[0].shape[0])) # add wn as first row

print(f"Data keys: {data.keys()}, Data shape: {data[str(sample) if sample is not None else 'si'].shape}, Xlsx shape: {xlsx_to_export.shape}")

xlsx_to_export[0, :] = data[str(sample) if sample is not None else 'si'][:, 0]  # Wavenumbers as first row

for key_id, key in enumerate(data.keys()):
  # xlsx_to_export[int(key_id)+1, :] = data[key][:, 4] # Provide relative error
  xlsx_to_export[int(key_id)+1, :] = data[key][:, 3] # Provide absolute error
  ax.plot(xlsx_to_export[0,:], np.abs(xlsx_to_export[int(key_id)+1, :]), label=str(key_id))          #
ax.set(
  xlim=(0, 120),
  ylim=(-1, 1),
)
ax.legend()

pd.DataFrame(xlsx_to_export).to_excel((ROOT / 'data/experimental_data/metastructure') / "difference_data2.xlsx", index=False)

In [ ]:
import sys

import torch
from pidu.data.utils import  generate_noise_profile_uc3, np2pt

from torch import float32 as pt_float32, device as pt_device

meas = pd.read_excel((ROOT / 'data/experimental_data/metastructure/difference_data2.xlsx'), sheet_name=None, index_col=0)['Sheet1'].to_numpy().T

wn = np2pt(meas[:,0], dtype=pt_float32, device=pt_device("cpu"))
bins = torch.arange(wn[0], wn[-1], 1, dtype=pt_float32, device=pt_device("cpu"))

_dict, wn_mask = generate_noise_profile_uc3(
  (ROOT / 'data/experimental_data/metastructure/difference_data2.xlsx'), 
  wn=wn, 
  bins=bins, 
  n_samples=meas.shape[0], 
  dtype=torch.float32, 
  device='cpu'
)

print(_dict['system1'].shape)

fig, ax = plt.subplots(1,1)